<a href="https://colab.research.google.com/github/Aksh6472/ML-data-analytics-/blob/main/Energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------
# 1. LOAD THE DATA
# -----------------------------------------------------------------------
INPUT_FILE = "/content/Energy consumption with nulls.csv"

df = pd.read_csv(INPUT_FILE)

In [3]:
print("=" * 60)
print("STEP 1: Raw data overview")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

STEP 1: Raw data overview
Shape: 1000 rows, 11 columns

Column data types:
Timestamp             object
Temperature          float64
Humidity             float64
SquareFootage        float64
Occupancy            float64
HVACUsage             object
LightingUsage         object
RenewableEnergy      float64
DayOfWeek             object
Holiday               object
EnergyConsumption    float64
dtype: object

First 5 rows:
          Timestamp  Temperature   Humidity  SquareFootage  Occupancy  \
0  01-01-2022 00:00    25.139433  43.431581    1565.693999        5.0   
1  01-01-2022 01:00    27.731651  54.225919    1411.064918        1.0   
2  01-01-2022 02:00    28.704277  58.907658    1755.715009        2.0   
3  01-01-2022 03:00    20.080469  50.371637    1452.316318        1.0   
4  01-01-2022 04:00    23.097359  51.401421    1094.130359        9.0   

  HVACUsage LightingUsage  RenewableEnergy  DayOfWeek Holiday  \
0        On           Off         2.774699     Monday      No   
1       

In [4]:
# 2. DIAGNOSE MISSING VALUES
print("\n" + "=" * 60)
print("STEP 2: Missing value report")
print("=" * 60)
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
print(missing_report[missing_report["missing_count"] > 0])


STEP 2: Missing value report
                 missing_count  missing_pct
Temperature                 50          5.0
Humidity                    50          5.0
SquareFootage               50          5.0
Occupancy                   50          5.0
HVACUsage                   50          5.0
LightingUsage               50          5.0
RenewableEnergy             50          5.0
DayOfWeek                   50          5.0
Holiday                     50          5.0


In [5]:
#3. HANDLE MISSING VALUES

print("\n" + "=" * 60)
print("STEP 3: Handling missing values")
print("=" * 60)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
categorical_cols.remove("Timestamp")  # Timestamp is text but not "categorical"

print(f"Numeric columns (median fill): {numeric_cols}")
print(f"Categorical columns (mode fill): {categorical_cols}")

for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  Filled '{col}' nulls with median = {median_val:.2f}")

for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  Filled '{col}' nulls with mode = '{mode_val}'")

print(f"\nRemaining nulls after cleaning: {df.isnull().sum().sum()}")



STEP 3: Handling missing values
Numeric columns (median fill): ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'EnergyConsumption']
Categorical columns (mode fill): ['HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday']
  Filled 'Temperature' nulls with median = 24.75
  Filled 'Humidity' nulls with median = 45.80
  Filled 'SquareFootage' nulls with median = 1507.77
  Filled 'Occupancy' nulls with median = 5.00
  Filled 'RenewableEnergy' nulls with median = 15.03
  Filled 'HVACUsage' nulls with mode = 'Off'
  Filled 'LightingUsage' nulls with mode = 'Off'
  Filled 'DayOfWeek' nulls with mode = 'Friday'
  Filled 'Holiday' nulls with mode = 'No'

Remaining nulls after cleaning: 0


In [6]:
# 4. FEATURE ENGINEERING FROM TIMESTAMP
print("\n" + "=" * 60)
print("STEP 4: Feature engineering from Timestamp")
print("=" * 60)

df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d-%m-%Y %H:%M")
df["Hour"] = df["Timestamp"].dt.hour
df["Month"] = df["Timestamp"].dt.month
print("Added 'Hour' and 'Month' columns extracted from Timestamp.")



STEP 4: Feature engineering from Timestamp
Added 'Hour' and 'Month' columns extracted from Timestamp.


In [7]:
#5. ENCODE CATEGORICAL VARIABLES

print("\n" + "=" * 60)
print("STEP 5: Encoding categorical variables")
print("=" * 60)

binary_maps = {
    "HVACUsage": {"On": 1, "Off": 0},
    "LightingUsage": {"On": 1, "Off": 0},
    "Holiday": {"Yes": 1, "No": 0}
}
for col, mapping in binary_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)
        print(f"  Encoded '{col}' as binary (1/0)")
    else:
        print(f"  Warning: Column '{col}' not found for binary encoding.")

# Check if 'DayOfWeek' column exists before attempting one-hot encoding
if "DayOfWeek" in df.columns:
    df = pd.get_dummies(df, columns=["DayOfWeek"], prefix="Day", drop_first=True)
    print("  One-hot encoded 'DayOfWeek' (drop_first=True to avoid redundancy)")
else:
    print("  Warning: 'DayOfWeek' column not found for one-hot encoding.")

# Drop the raw Timestamp now that we've extracted Hour/Month from it
if "Timestamp" in df.columns:
    df = df.drop(columns=["Timestamp"])
else:
    print("  Warning: 'Timestamp' column not found for dropping.")


STEP 5: Encoding categorical variables
  Encoded 'HVACUsage' as binary (1/0)
  Encoded 'LightingUsage' as binary (1/0)
  Encoded 'Holiday' as binary (1/0)
  One-hot encoded 'DayOfWeek' (drop_first=True to avoid redundancy)


In [9]:
OUTPUT_FILE = "/content/cleaned_energy_data.csv"

# 6. SAVE CLEANED DATASET

print("\n" + "=" * 60)
print("STEP 6: Saving cleaned dataset")
print("=" * 60)
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Final columns: {list(df.columns)}")

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved cleaned data to '{OUTPUT_FILE}'")
print("This file is used as input for T2 (linear regression).")


STEP 6: Saving cleaned dataset
Final shape: 1000 rows, 17 columns
Final columns: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Day_Monday', 'Day_Saturday', 'Day_Sunday', 'Day_Thursday', 'Day_Tuesday', 'Day_Wednesday']

Saved cleaned data to '/content/cleaned_energy_data.csv'
This file is used as input for T2 (linear regression).


T2: Building programs to work with linear regression in Python

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [12]:
INPUT_FILE = "/content/cleaned_energy_data.csv"
df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("STEP 1: Cleaned data overview")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumns:", list(df.columns))

STEP 1: Cleaned data overview
Shape: 1000 rows, 17 columns

Columns: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Day_Monday', 'Day_Saturday', 'Day_Sunday', 'Day_Thursday', 'Day_Tuesday', 'Day_Wednesday']


In [14]:
print("\n" + "=" * 60)
print("STEP 2: Splitting features and target")
print("=" * 60)
TARGET = "EnergyConsumption"
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f"Target: {TARGET}")
print(f"Features ({X.shape[1]}): {list(X.columns)}")


STEP 2: Splitting features and target
Target: EnergyConsumption
Features (16): ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'Holiday', 'Hour', 'Month', 'Day_Monday', 'Day_Saturday', 'Day_Sunday', 'Day_Thursday', 'Day_Tuesday', 'Day_Wednesday']


In [16]:
print("\n" + "=" * 60)
print("STEP 3: Train/test split")
print("=" * 60)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")


STEP 3: Train/test split
Train set: 800 rows
Test set:  200 rows


In [17]:
print("\n" + "=" * 60)
print("STEP 4: Training Linear Regression model")
print("=" * 60)
model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained.")


STEP 4: Training Linear Regression model
Model trained.


In [19]:
print("\n" + "=" * 60)
print("STEP 5: Making predictions on test set")
print("=" * 60)
y_pred = model.predict(X_test)
print("Sample predictions vs actual:")
comparison = pd.DataFrame({"Actual": y_test.values[:5], "Predicted": y_pred[:5]})
print(comparison)


STEP 5: Making predictions on test set
Sample predictions vs actual:
      Actual  Predicted
0  86.920611  81.423266
1  88.351606  76.289268
2  79.431363  75.364059
3  90.009188  78.029321
4  83.891100  80.601233


In [21]:
print("\n" + "=" * 60)
print("STEP 6: Model evaluation")
print("=" * 60)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE  (Mean Absolute Error) : {mae:.3f}")
print(f"MSE  (Mean Squared Error)  : {mse:.3f}")
print(f"RMSE (Root Mean Sq. Error) : {rmse:.3f}")
print(f"R2   (R-squared)           : {r2:.3f}")



STEP 6: Model evaluation
MAE  (Mean Absolute Error) : 4.647
MSE  (Mean Squared Error)  : 33.871
RMSE (Root Mean Sq. Error) : 5.820
R2   (R-squared)           : 0.483


In [23]:
print("\n" + "=" * 60)
print("STEP 7: Feature coefficients")
print("=" * 60)
coefficients = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(coefficients)
print(f"\nIntercept: {model.intercept_:.3f}")
print("\nThis model is used as input for T3 (cross validation).")


STEP 7: Feature coefficients
HVACUsage          4.150685
Temperature        1.993818
LightingUsage      1.358832
Day_Saturday       0.860682
Holiday            0.749477
Day_Tuesday        0.721845
Day_Thursday       0.605288
Day_Sunday         0.597319
Occupancy          0.515066
Day_Monday         0.504565
Month              0.492837
Day_Wednesday      0.139452
RenewableEnergy    0.100094
Humidity          -0.040913
Hour              -0.033419
SquareFootage     -0.000243
dtype: float64

Intercept: 21.970

This model is used as input for T3 (cross validation).


T3: cross validation


In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
INPUT_FILE = "/content/cleaned_energy_data.csv"
df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("STEP 1: Cleaned data overview")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

STEP 1: Cleaned data overview
Shape: 1000 rows, 17 columns


In [27]:
print("\n" + "=" * 60)
print("STEP 2: Splitting features and target")
print("=" * 60)
TARGET = "EnergyConsumption"
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f"Target: {TARGET}")
print(f"Features ({X.shape[1]}): {list(X.columns)}")


STEP 2: Splitting features and target
Target: EnergyConsumption
Features (16): ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'Holiday', 'Hour', 'Month', 'Day_Monday', 'Day_Saturday', 'Day_Sunday', 'Day_Thursday', 'Day_Tuesday', 'Day_Wednesday']


In [29]:
print("\n" + "=" * 60)
print("STEP 3: Setting up K-Fold cross validation")
print("=" * 60)
K = 5
kfold = KFold(n_splits=K, shuffle=True, random_state=42)
print(f"Number of folds: {K}")

model = LinearRegression()


STEP 3: Setting up K-Fold cross validation
Number of folds: 5


In [30]:
print("\n" + "=" * 60)
print("STEP 4: Cross validation - R2 score per fold")
print("=" * 60)
r2_scores = cross_val_score(model, X, y, cv=kfold, scoring="r2")
for i, score in enumerate(r2_scores, start=1):
    print(f"  Fold {i}: R2 = {score:.3f}")
print(f"\nMean R2 : {r2_scores.mean():.3f}")
print(f"Std  R2 : {r2_scores.std():.3f}")



STEP 4: Cross validation - R2 score per fold
  Fold 1: R2 = 0.483
  Fold 2: R2 = 0.625
  Fold 3: R2 = 0.623
  Fold 4: R2 = 0.503
  Fold 5: R2 = 0.606

Mean R2 : 0.568
Std  R2 : 0.062


In [32]:
print("\n" + "=" * 60)
print("STEP 5: Cross validation - MAE per fold")
print("=" * 60)
mae_scores = -cross_val_score(model, X, y, cv=kfold, scoring="neg_mean_absolute_error")
for i, score in enumerate(mae_scores, start=1):
    print(f"  Fold {i}: MAE = {score:.3f}")
print(f"\nMean MAE : {mae_scores.mean():.3f}")
print(f"Std  MAE : {mae_scores.std():.3f}")


STEP 5: Cross validation - MAE per fold
  Fold 1: MAE = 4.647
  Fold 2: MAE = 4.089
  Fold 3: MAE = 4.037
  Fold 4: MAE = 4.245
  Fold 5: MAE = 4.379

Mean MAE : 4.279
Std  MAE : 0.219


In [35]:
print("\n" + "=" * 60)
print("STEP 6: Cross validation - RMSE per fold")
print("=" * 60)
mse_scores = -cross_val_score(model, X, y, cv=kfold, scoring="neg_mean_squared_error")
rmse_scores = np.sqrt(mse_scores)
for i, score in enumerate(rmse_scores, start=1):
    print(f"  Fold {i}: RMSE = {score:.3f}")
print(f"\nMean RMSE : {rmse_scores.mean():.3f}")
print(f"Std  RMSE : {rmse_scores.std():.3f}")



STEP 6: Cross validation - RMSE per fold
  Fold 1: RMSE = 5.820
  Fold 2: RMSE = 5.131
  Fold 3: RMSE = 4.882
  Fold 4: RMSE = 5.330
  Fold 5: RMSE = 5.383

Mean RMSE : 5.309
Std  RMSE : 0.310


In [36]:
print("\n" + "=" * 60)
print("STEP 7: Summary")
print("=" * 60)
summary = pd.DataFrame({
    "R2": r2_scores,
    "MAE": mae_scores,
    "RMSE": rmse_scores
}, index=[f"Fold {i}" for i in range(1, K + 1)])
print(summary)
print("\nAverages:")
print(summary.mean())
print("\nA small std across folds means the model performs consistently")
print("on different subsets of data (i.e. it is not overfitting to one split).")


STEP 7: Summary
              R2       MAE      RMSE
Fold 1  0.482890  4.646625  5.819846
Fold 2  0.624663  4.089107  5.131395
Fold 3  0.623260  4.037055  4.882109
Fold 4  0.503415  4.245306  5.330196
Fold 5  0.606133  4.378557  5.383178

Averages:
R2      0.568072
MAE     4.279330
RMSE    5.309345
dtype: float64

A small std across folds means the model performs consistently
on different subsets of data (i.e. it is not overfitting to one split).
